# iNaturalist Reliability Workbench

Notebook starter for `pynat.reliability`.

Suggested workflow:
1. Set your user login and cache path.
2. Run ingest once (or when you want fresh source data).
3. Iterate offline with different filters and custom charts.

In [10]:
from pathlib import Path
import importlib
import sys

import pandas as pd

# Find the repository root (folder containing reliability.py) and import from there.
nb_dir = Path.cwd()
repo_root = next((p for p in [nb_dir, *nb_dir.parents] if (p / "reliability.py").exists()), nb_dir)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import reliability
importlib.reload(reliability)
Analyzer = reliability.Analyzer
DEFAULT_CACHE_DIR = reliability.DEFAULT_CACHE_DIR
print(f"Loaded reliability from: {Path(reliability.__file__).resolve()}")

Loaded reliability from: C:\Users\drsvs\Desktop\code\pynat\reliability.py


In [11]:
USER_LOGIN = "schizoform"
CACHE_DIR = DEFAULT_CACHE_DIR

print(f"Using cache dir: {Path(CACHE_DIR).resolve()}")
analyzer = Analyzer(cache_dir=CACHE_DIR)

Using cache dir: C:\Users\drsvs\Desktop\code\pynat\data


In [12]:
# Lightweight one-user + one-taxon assessment (no cache ingest required)
TAXON_ID = 130953  # replace with your target taxon id

# Optional date filters (YYYY-MM-DD). Keep as None for all-time scope.
START_DATE = None
END_DATE = None

out_taxon = analyzer.assess_taxon(
    user_login=USER_LOGIN,
    taxon_id=TAXON_ID,
    start=START_DATE,
    end=END_DATE,
    include_withdrawn=True,
    print_report=False,
)

df_taxon_props = out_taxon["proposals"]
df_taxon_rel = out_taxon["taxon_reliability"]
meta = out_taxon["analysis_meta"]

target_name = meta.get("target_taxon_name") or "(unknown scientific name)"
target_id = meta.get("target_taxon_id", TAXON_ID)
print(f"Assessment target: {target_name} ({target_id})")
print(
    "Taxon scope: target taxon plus descendants; "
    f"include_withdrawn={meta.get('include_withdrawn')}, "
    f"candidate IDs={meta.get('candidate_identification_count', 'n/a')}, "
    f"lineage-matching IDs={meta.get('lineage_filtered_identification_count', 'n/a')}, "
    f"observations touched={meta.get('observation_scope_count', 'n/a')}."
)
print(
    "Generated proposal table: "
    f"rows={meta.get('proposal_rows_generated', len(df_taxon_props))}, "
    f"unique observations={meta.get('proposal_observation_count', df_taxon_props['obs_id'].nunique() if not df_taxon_props.empty else 0)}, "
    f"unique proposed taxa={meta.get('unique_proposed_taxa_count', df_taxon_props['taxon_id'].nunique() if not df_taxon_props.empty else 0)}."
 )
print(f"Taxonomic reliability rows generated: {len(df_taxon_rel)}")
print(f"Partial API coverage: {meta.get('partial_results')}")
print(f"Loaded proposal count from API windowing: {meta.get('loaded_proposal_count')}")
print(f"Oldest loaded proposal timestamp: {meta.get('oldest_loaded_proposed_at')}")
if meta.get("warning"):
    print("warning:", meta.get("warning"))

display_cols = [
    "obs_id",
    "proposed_taxon",
    "proposed_taxon_id",
    "community_taxon",
    "community_taxon_id",
    "confirmed_status",
    "outcome",
    "taxonomic_level",
    "community_taxonomic_level",
    "community_overlap_depth",
]
available_cols = [c for c in display_cols if c in df_taxon_props.columns]
missing_cols = [c for c in display_cols if c not in df_taxon_props.columns]
if missing_cols:
    print(f"Missing expected columns: {missing_cols}")

preview = df_taxon_props[available_cols].head(30)

# Focus rows mentioned during review
focus_obs = [116362962, 160260924]
focus_rows = df_taxon_props[df_taxon_props["obs_id"].isin(focus_obs)][available_cols]
print("Focus observations (subset):")
print(focus_rows.sort_values(["obs_id", "proposed_taxon_id"], ascending=[True, True]))

preview

[assess_taxon:scope] Fetching identifications for user 'schizoform' within taxon 130953 (target + descendants).
[assess_taxon:scope] Retained 31 proposals across 29 observations for target lineage.
[assess_taxon:timeline] Fetching full identification timelines for scoped observations.
observations: 100%|██████████| 1/1 [00:00<00:00,  1.97it/s]
[assess_taxon:build] Building proposal outcomes from observation timelines.
proposals(obs): 100%|██████████| 29/29 [00:00<00:00, 87.73it/s]

Assessment target: Nabalus (130953)
Taxon scope: target taxon plus descendants; include_withdrawn=True, candidate IDs=31, lineage-matching IDs=31, observations touched=29.
Generated proposal table: rows=31, unique observations=29, unique proposed taxa=3.
Taxonomic reliability rows generated: 5
Partial API coverage: False
Loaded proposal count from API windowing: 31
Oldest loaded proposal timestamp: 2019-08-15T20:11:07+00:00
Focus observations (subset):
       obs_id       proposed_taxon  proposed_taxon_id community_taxon  \
12  116362962  Nabalus serpentaria             204361            <NA>   
18  160260924              Nabalus             130953            <NA>   

    community_taxon_id       confirmed_status     outcome taxonomic_level  \
12                <NA>  no_confirming_id_seen  unresolved           genus   
18               47124  no_confirming_id_seen  unresolved           genus   

   community_taxonomic_level community_overlap_depth  
12                      <NA>        


[assess_taxon:build] Built 31 proposals for downstream summaries.


,obs_id,proposed_taxon,proposed_taxon_id,community_taxon,community_taxon_id,confirmed_status,outcome,taxonomic_level,community_taxonomic_level,community_overlap_depth
0,6210424,Nabalus,130953,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
1,45498435,Nabalus,130953,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
2,68073886,Nabalus,130953,<NA>,359450,no_confirming_id_seen,unresolved,genus,<NA>,family
3,78653051,Nabalus trifoliolatus,204185,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
4,78653051,Nabalus serpentaria,204361,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
5,79208300,Nabalus,130953,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
6,79208300,Nabalus serpentaria,204361,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
7,109563596,Nabalus serpentaria,204361,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
8,110801784,Nabalus serpentaria,204361,<NA>,<NA>,no_confirming_id_seen,unresolved,genus,<NA>,no_ct
9,112733058,Nabalus serpentaria,204361,<NA>,204361,confirmed_after_proposal,confirmed,genus,<NA>,species


In [4]:
df_taxon_rel.sort_values(["confirmed_to_disconfirmed_ratio", "n_props"], ascending=[False, False])[
    [
        "target_taxon_label",
        "taxonomic_level",
        "n_props",
        "confirmed_count",
        "disconfirmed_count",
        "unresolved_count",
        "confirmed_to_disconfirmed_ratio",
    ]
]

,target_taxon_label,taxonomic_level,n_props,confirmed_count,disconfirmed_count,unresolved_count,confirmed_to_disconfirmed_ratio
3,Nabalus (130953),genus,26,5,0,21,5.0
1,Magnoliopsida (47124),class,2,0,0,2,0.0
0,Plantae (47126),kingdom,0,0,0,0,0.0
2,Cichorieae (359450),tribe,0,0,0,0,0.0


In [5]:
# Quick validation helper: compare proposal rows and touched observations
validation = {
    "target_taxon": f"{target_name} ({target_id})",
    "include_withdrawn": bool(meta.get("include_withdrawn")),
    "lineage_matching_user_ids": int(meta.get("lineage_filtered_identification_count", 0) or 0),
    "proposal_rows": int(len(df_taxon_props)),
    "observations_touched": int(df_taxon_props["obs_id"].nunique()) if not df_taxon_props.empty else 0,
    "community_taxa_seen": int(df_taxon_props["community_taxon_id"].nunique()) if not df_taxon_props.empty else 0,
    "community_ranks_seen": int(df_taxon_props["community_taxonomic_level"].nunique()) if (not df_taxon_props.empty and "community_taxonomic_level" in df_taxon_props.columns) else 0,
}
pd.Series(validation, name="validation")

target_taxon                 Nabalus (130953)
include_withdrawn                        True
lineage_matching_user_ids                  31
proposal_rows                              28
observations_touched                       28
community_taxa_seen                         3
community_ranks_seen                        0
Name: validation, dtype: object

taxon 130953: name=Nabalus, rank=genus, ancestor_count=11
  includes target as ancestor? True
  first ancestors: [48460, 47126, 211194, 47125, 47124, 47605, 47604, 447425, 359450, 632701, 130953]
taxon 461542: name=Astereae, rank=tribe, ancestor_count=9
  includes target as ancestor? False
  first ancestors: [48460, 47126, 211194, 47125, 47124, 47605, 47604, 632790, 461542]


,obs_id,taxon_id,proposed_taxon,taxonomic_level,community_taxon,community_taxonomic_level
10,116362962,461542,Astereae,class,<NA>,<NA>
